# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

The dataset contains ordered logistic regression outputs related to knowledge adoption for development interventions among pastoralist households in Northern Kenya.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Install mlcroissant if not already present
!pip install mlcroissant --quiet

## 1. Data Loading

Load the dataset metadata using `mlcroissant`. This also checks Croissant schema compliance.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview

Next, we'll list available record sets and their fields, referencing them by their `@id` as required by Croissant. Each record set and field `@id` can be used for future queries and data extraction.

In [ ]:
print("Available Record Sets in the dataset:")
for record_set in dataset.record_sets:
    print(f"- Record Set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}\n      Name: {field.name}\n      Data type: {field.data_type}")
    print()

## 3. Data Extraction

We'll now extract data from one or more record sets using their `@id` values. Each record set maps to a structured table, and extraction yields a DataFrame for convenient analysis.

**Steps:**
1. Identify one or more record sets by their `@id`.
2. Load records from each record set into a DataFrame.
3. Inspect the columns (`@id`s of fields) for further exploration.

In [ ]:
# List to keep track of available record set IDs
record_set_ids = [rs.id for rs in dataset.record_sets]

# Load all available record sets into separate DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set: {record_set_id} with {len(records)} rows")

# Display columns for each record set
for record_set_id, df in dataframes.items():
    print(f"\nColumns for record set {record_set_id}:")
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

This step demonstrates typical data processing — filtering on a numeric field, normalization, and simple grouping. We'll select a numeric field from the first loaded record set (by `@id`) as an example.

> **Note:** Fields and groups are always referenced by their Croissant `@id`.

In [ ]:
# Choose the first record set for demonstration
if len(dataframes) == 0:
    raise ValueError('No record sets loaded from the Croissant dataset.')

record_set_id = record_set_ids[0]
df = dataframes[record_set_id]

# Find a numeric field (we'll choose the first float/int field present)
numeric_field_id = None
group_field_id = None

for record_set in dataset.record_sets:
    if record_set.id == record_set_id:
        for field in record_set.fields:
            if field.data_type and field.data_type.lower() in ['float', 'integer', 'number']:
                numeric_field_id = field.id
            elif not group_field_id and field.data_type and field.data_type.lower() in ['text', 'string']:
                group_field_id = field.id
        break

print(f"Using record set @id: {record_set_id}")
print(f"Numeric field @id: {numeric_field_id}")
print(f"Grouping field @id: {group_field_id}")

if numeric_field_id not in df.columns:
    print(f'No numeric field found in {record_set_id}. Skipping filtering and normalization.')
else:
    # Example thresholding
    threshold = 10  # Arbitrary threshold for demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} ({len(filtered_df)} rows):")
    display(filtered_df.head())

    # Normalize the numeric field (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) 
        / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized field for {numeric_field_id}:")
    display(filtered_df[[numeric_field_id,f"{numeric_field_id}_normalized"]].head())

    # Group by some categorical field if present (just an example, e.g., a location or ward)
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped.head())

## 5. Visualization

Visualize data distributions or feature relationships. Below we'll plot the distribution of our chosen numeric field and visualize group differences (if those columns exist in the data).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for numeric field, if available:
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of field '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # Boxplot by group, if grouping is possible
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

- We've successfully loaded the Croissant FAIR² dataset and programmatically explored its structure and content using `mlcroissant`.
- Data fields and entities were referenced using their `@id`, ensuring reproducibility.
- Pipeline included loading, field inspection, EDA (with basic filtering and normalization), and visualization.

You can adapt this notebook to interrogate other record sets, fields, or to apply richer analyses!

**Reference**: [FAIR2 Dataset Package](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

[mlcroissant Documentation](https://github.com/mlcommons/croissant)
